In [1]:
import sys
import os

path_to_scripts = os.path.join('..', '..', 'python_scripts')
sys.path.append(path_to_scripts)

%load_ext autoreload

In [7]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_error, max_error

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit

from data_to_bigquery import load_from_bigquery
from feature_engineering import drop_lag_nulls, validate_features, engineer_features
from baseline_model import baseline_model_xgb, xgb_train_preproc, evaluate_trained_model, pred_preproc_xgb, xgb_prediction
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

from statsmodels.graphics.tsaplots import plot_acf

%matplotlib inline

In [4]:
 # loading up to date df with cyclical feature eng
df = load_from_bigquery('gridzero-489711','merged_set','full_feature_engineered_data_test')


/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 148,991 rows and 31 columns from gridzero-489711.merged_set.full_feature_engineered_data_test


In [8]:
# preproc new splitting
# test on 2026
# train on 2025 ONLY
# defining X and target
# keeping dattime to help plotting
# still splitting temporally bc multiple years

target_col = 'carbon_intensity_gco2_kwh'

# sort by datetime and reset index ooooo
df = df.sort_values('datetime').reset_index(drop=True)

target_col = 'carbon_intensity_gco2_kwh'

# temporal split
train_df = df[df['datetime'].dt.year < 2025]
test_df  = df[df['datetime'].dt.year >= 2025]

X_train = train_df.drop(columns=[target_col, 'datetime'])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col, 'datetime'])
y_test = test_df[target_col]

# keep only num cola to make xgboost happy
feature_cols = X_train.select_dtypes(include='number').columns.tolist()

X_train = X_train[feature_cols]
X_test = X_test[feature_cols]

print(X_train.shape, X_test.shape)

(128064, 29) (20927, 29)


In [12]:
df.head()
cols = df.columns.tolist()
cols

['datetime',
 'temperature_2m_c',
 'wind_speed_100m_ms',
 'wind_gusts_10m_ms',
 'cloud_cover_pct',
 'shortwave_radiation_wm2',
 'direct_radiation_wm2',
 'diffuse_radiation_wm2',
 'pressure_msl_hpa',
 'precipitation_mm',
 'biomass',
 'fossil_gas',
 'fossil_hard_coal',
 'hydro_pumped_storage',
 'hydro_run_of_river_and_poundage',
 'nuclear',
 'other',
 'solar',
 'wind_offshore',
 'wind_onshore',
 'totaloutput_mw',
 'carbon_intensity_gco2_kwh',
 'hour_sin',
 'hour_cos',
 'dow_sin',
 'dow_cos',
 'doy_sin',
 'doy_cos',
 'carbon_lag_48',
 'carbon_lag_336',
 'carbon_lag_17520']

In [ ]:
# defining target and features columns
target_cols = df[[
                'biomass',
                'fossil_gas',
                'fossil_hard_coal',
                'hydro_pumped_storage',
                'hydro_run_of_river_and_poundage',
                'nuclear',
                'other',
                'solar',
                'wind_offshore',
                'wind_onshore',
                'totaloutput_mw'
                ]]

feature_cols = df.drop(columns=target_cols).columns.tolist()